# Getting Started – Data Engineering with Snowflake

Build an end-to-end **Ingestion → Transformation → Delivery (I-T-D)** pipeline for **Tasty Bytes**, a global food-truck company. Analysts flagged that **sales in Hamburg, Germany dropped to $0 for several days in February 2022** — this pipeline finds out why.

You'll run **every stage in this notebook**. Some cells are pre-written; others you'll generate by prompting **CoCo** (the prompt is shown in the markdown cell right above each one).

**Layers:** `RAW_POS` (Ingestion) → `HARMONIZED` (Transformation) → `ANALYTICS` (Delivery: Semantic View, Streamlit dashboard, Cortex Analyst).

## Connect this notebook to compute

This notebook runs inside a **Git-backed Workspace**. Before running any cell:

1. Click **Connect** next to the Run button and select **Create and connect**. You can monitor the connection status in the status bar at the bottom of the notebook — wait for it to show **Connected** before proceeding.

2. Set your **role** and **warehouse** using the picker at the top-left of the Notebooks editor:
   - **Role = `ACCOUNTADMIN`**
   - **Warehouse = `COMPUTE_WH`** (the default trial warehouse)

   This is the notebook's **query warehouse** — every SQL cell and any Snowpark pushdown runs on it. (Rendering the results grid is free.) You can also set these in a cell with `USE ROLE` / `USE WAREHOUSE`.

3. Notebooks in Workspaces **don't auto-select a database or schema**, so the cells below set context explicitly (`USE DATABASE` / `USE SCHEMA`) and use fully qualified names — objects resolve wherever you run them.

4. Run cells top-to-bottom with **▶** / **Shift+Enter**, or **Run all** from the toolbar. If the session idle-suspends (~30 min of inactivity), restart it from the top-left and re-run from where you left off — created objects persist in Snowflake.

Where a markdown cell shows a **CoCo prompt**, open the **CoCo** panel, paste the prompt, compare CoCo's output against the **Expected output** in the cell, and click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

In [ ]:
-- SETUP — create the three-layer database and enable Cortex AI features.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE tasty_bytes;
CREATE OR REPLACE SCHEMA tasty_bytes.raw_pos;
CREATE OR REPLACE SCHEMA tasty_bytes.harmonized;
CREATE OR REPLACE SCHEMA tasty_bytes.analytics;

-- Allows CoCo and Cortex Analyst to use models from any region
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## Ingestion — install the live weather share (programmatically)

The first data source is **live weather** from Snowflake Marketplace (Pelmorex Weather Source: Frostbyte). Instead of clicking through the Marketplace UI, we install it programmatically: accept the listing's legal terms, then create a database directly from the listing. This is the repeatable, scriptable way to acquire a Marketplace dataset.

In [ ]:
-- Accept the listing terms and install the Pelmorex weather share as FROSTBYTE_WEATHERSOURCE
USE ROLE ACCOUNTADMIN;

CALL SYSTEM$ACCEPT_LEGAL_TERMS('DATA_EXCHANGE_LISTING', 'GZSOZ1LLEL');

CREATE DATABASE IF NOT EXISTS FROSTBYTE_WEATHERSOURCE
  FROM LISTING 'GZSOZ1LLEL';

## Ingestion — create the raw POS tables and ORDERS_V (boilerplate)

Run the next cell as-is. It creates the empty raw POS target tables and the `ANALYTICS.ORDERS_V` view — a denormalized flat join across the raw tables that gives every downstream query one simple source for sales. There's nothing to "solve" here.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE DATABASE tasty_bytes;
USE SCHEMA raw_pos;

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.country (
    country_id       NUMBER(18,0),
    country          VARCHAR(16777216),
    iso_country      VARCHAR(16777216),
    city             VARCHAR(16777216),
    city_population  VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.franchise (
    franchise_id  NUMBER(38,0),
    first_name    VARCHAR(16777216),
    last_name     VARCHAR(16777216),
    city          VARCHAR(16777216),
    country       VARCHAR(16777216),
    e_mail        VARCHAR(16777216),
    phone_number  VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.location (
    location_id       NUMBER(19,0),
    placekey          VARCHAR(16777216),
    location          VARCHAR(16777216),
    city              VARCHAR(16777216),
    region            VARCHAR(16777216),
    iso_country_code  VARCHAR(16777216),
    country           VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.menu (
    menu_id                        NUMBER(19,0),
    menu_type_id                   NUMBER(38,0),
    menu_type                      VARCHAR(16777216),
    truck_brand_name               VARCHAR(16777216),
    menu_item_id                   NUMBER(38,0),
    menu_item_name                 VARCHAR(16777216),
    item_category                  VARCHAR(16777216),
    item_subcategory               VARCHAR(16777216),
    cost_of_goods_usd              NUMBER(38,4),
    sale_price_usd                 NUMBER(38,4),
    menu_item_health_metrics_obj   VARIANT
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck (
    truck_id             NUMBER(38,0),
    menu_type_id         NUMBER(18,0),
    primary_city         VARCHAR(16777216),
    region               VARCHAR(16777216),
    iso_region           VARCHAR(16777216),
    country              VARCHAR(16777216),
    iso_country_code     VARCHAR(16777216),
    franchise_flag       NUMBER(38,0),
    year                 NUMBER(38,0),
    make                 VARCHAR(16777216),
    model                VARCHAR(16777216),
    ev_flag              NUMBER(38,0),
    franchise_id         NUMBER(38,0),
    truck_opening_date   DATE
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_header (
    order_id                NUMBER(38,0),
    truck_id                NUMBER(38,0),
    location_id             NUMBER(19,0),
    customer_id             NUMBER(38,0),
    discount_id             VARCHAR(16777216),
    shift_id                NUMBER(38,0),
    shift_start_time        TIME(9),
    shift_end_time          TIME(9),
    order_channel           VARCHAR(16777216),
    order_ts                TIMESTAMP_NTZ(9),
    served_ts               VARCHAR(16777216),
    order_currency          VARCHAR(3),
    order_amount            NUMBER(38,4),
    order_tax_amount        VARCHAR(16777216),
    order_discount_amount   VARCHAR(16777216),
    order_total             NUMBER(38,4)
);

CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_detail (
    order_detail_id              NUMBER(38,0),
    order_id                     NUMBER(38,0),
    menu_item_id                 NUMBER(38,0),
    discount_id                  VARCHAR(16777216),
    line_number                  NUMBER(38,0),
    quantity                     NUMBER(5,0),
    unit_price                   NUMBER(38,4),
    price                        NUMBER(38,4),
    order_item_discount_amount   VARCHAR(16777216)
);

-- Denormalized flat sales view used by downstream steps and the semantic view.
CREATE OR REPLACE VIEW tasty_bytes.analytics.orders_v
AS
SELECT
    oh.order_id,
    oh.truck_id,
    oh.order_ts,
    od.order_detail_id,
    od.line_number,
    m.truck_brand_name,
    m.menu_type,
    t.primary_city,
    t.region,
    t.country,
    t.franchise_flag,
    t.franchise_id,
    f.first_name  AS franchisee_first_name,
    f.last_name   AS franchisee_last_name,
    l.location,
    oh.order_channel,
    oh.order_amount AS price,
    oh.order_total
FROM tasty_bytes.raw_pos.order_header  oh
JOIN tasty_bytes.raw_pos.order_detail   od ON oh.order_id      = od.order_id
JOIN tasty_bytes.raw_pos.truck           t  ON oh.truck_id     = t.truck_id
JOIN tasty_bytes.raw_pos.menu            m  ON od.menu_item_id = m.menu_item_id
JOIN tasty_bytes.raw_pos.franchise       f  ON t.franchise_id  = f.franchise_id
JOIN tasty_bytes.raw_pos.location        l  ON oh.location_id  = l.location_id;

### STEP 1 — Create a CSV file format

A **file format** tells Snowflake how to parse raw files during loading. We need one for the CSV data on S3.

Open the **CoCo** panel and send this prompt. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Create a CSV file format named CSV_FF in TASTY_BYTES.PUBLIC with type = 'csv'."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Create a CSV file format" | Asks for a reusable parsing definition, not a table or a data load |
| "named `CSV_FF` in `TASTY_BYTES.PUBLIC`" | Gives the exact name and location. The stage and later loads reference this object. |
| "type = 'csv'" | Specifies the source file type, so CoCo doesn't have to choose a format |

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE FILE FORMAT TASTY_BYTES.PUBLIC.CSV_FF
  TYPE = 'CSV';
```
</details>

### STEP 2 — Create the external stage

A **stage** is a pointer to an external storage location (in this case, an S3 bucket) so Snowflake knows where to find the files.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Create an external stage named S3LOAD in TASTY_BYTES.PUBLIC that points to 's3://sfquickstarts/tastybytes/' and uses the CSV_FF file format."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Create an external stage named `S3LOAD` in `TASTY_BYTES.PUBLIC`" | Names the storage pointer that every `COPY INTO` statement will use |
| "points to 's3://sfquickstarts/tastybytes/'" | Gives the bucket and base path. Each table's load adds its own subfolder to this path. |
| "uses the `CSV_FF` file format" | Connects the stage to the parsing definition you just created, rather than asking CoCo to invent another one |

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE STAGE TASTY_BYTES.PUBLIC.S3LOAD
  URL = 's3://sfquickstarts/tastybytes/'
  FILE_FORMAT = TASTY_BYTES.PUBLIC.CSV_FF;
```
</details>

### STEP 3 — Load the COUNTRY table (teaching example)

`COPY INTO` is Snowflake's bulk-loading command — it reads files from a stage and inserts them into a table. This single load teaches you the pattern you'll repeat for every table.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. You should see ~30 rows load. **This single COPY INTO is the pattern for every table** — only the stage path and table name change. Optionally, copy the SQL into the notebook cell for future reference.

> *"Write a COPY INTO statement that loads data from @tasty_bytes.public.s3load/raw_pos/country/ into TASTY_BYTES.RAW_POS.COUNTRY. Use FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE) to handle extra columns in the source file."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Write a `COPY INTO` statement" | Specifies the bulk-loading operation, rather than creating another empty table |
| "from `@tasty_bytes.public.s3load/raw_pos/country/`" | Targets the COUNTRY files, not every CSV under the stage |
| "into `TASTY_BYTES.RAW_POS.COUNTRY`" | Names the existing destination table in the Ingestion layer |
| "`FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF`" | Reuses the same CSV parsing definition as the stage |
| "`ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE`" | Explicitly allows the column-count difference described in the prompt. It does not ask CoCo to ignore every kind of loading error. |

<details>
<summary>Expected output (check against this)</summary>

```sql
COPY INTO TASTY_BYTES.RAW_POS.COUNTRY
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/country/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);
```
</details>

### STEP 4 — Load all remaining tables (scale-up)

Now we apply the same `COPY INTO` pattern at scale — loading six more tables in one shot. CoCo will also spin up a larger warehouse for performance, then tear it down to avoid idle credit burn.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Load the remaining Tasty Bytes tables from the S3 stage @tasty_bytes.public.s3load into their corresponding tables in TASTY_BYTES.RAW_POS: FRANCHISE (raw_pos/franchise/), LOCATION (raw_pos/location/), MENU (raw_pos/menu/), TRUCK (raw_pos/truck/), ORDER_HEADER (raw_pos/order_header/), ORDER_DETAIL (raw_pos/order_detail/). First create a dedicated LARGE warehouse named LOAD_WH with AUTO_SUSPEND = 60 and AUTO_RESUME = TRUE, switch to it, run the COPY INTOs using FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE), then drop LOAD_WH and switch back to COMPUTE_WH."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| The six table names and their `raw_pos/.../` paths | Pairs each destination with its source folder, so CoCo doesn't have to guess which files belong in which table |
| "from the S3 stage `@tasty_bytes.public.s3load` … in `TASTY_BYTES.RAW_POS`" | Keeps the loads on the stage you created and in the raw Ingestion layer |
| "First create a dedicated LARGE warehouse named `LOAD_WH` … switch to it" | Requests separate compute for the larger load and makes the execution order explicit |
| "`AUTO_SUSPEND = 60` and `AUTO_RESUME = TRUE`" | Specifies the idle-suspension and restart settings for the temporary warehouse |
| "run the `COPY INTO`s using `FILE_FORMAT = ...`" | Applies the same CSV format and column-count option to all six loads |
| "then drop `LOAD_WH` and switch back to `COMPUTE_WH`" | Makes cleanup and restoring the notebook's usual warehouse part of the task, not something the reader has to remember later |

> **Note:** This load covers ~1 GB of data across 6 tables and may take several minutes. Wait for all success messages before continuing.

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE WAREHOUSE LOAD_WH
  WAREHOUSE_SIZE = 'LARGE'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE LOAD_WH;

COPY INTO TASTY_BYTES.RAW_POS.FRANCHISE
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/franchise/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.LOCATION
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/location/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.MENU
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/menu/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.TRUCK
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/truck/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_HEADER
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_header/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

COPY INTO TASTY_BYTES.RAW_POS.ORDER_DETAIL
FROM @TASTY_BYTES.PUBLIC.S3LOAD/raw_pos/order_detail/
FILE_FORMAT = (FORMAT_NAME = TASTY_BYTES.PUBLIC.CSV_FF ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE);

DROP WAREHOUSE LOAD_WH;
USE WAREHOUSE COMPUTE_WH;
```
</details>

## Transformation — UDFs and Dynamic Tables

Ingestion is done. Now we shape the raw data into analyst-ready tables using **SQL UDFs** and **Dynamic Tables**.

**About UDFs:** a SQL UDF **must return a value** of the type in its `RETURNS` clause — its body is a single expression whose result is returned on every call. That's what makes UDFs great for **reusable data transformations** (unit conversions, formatting, business math) you can invoke anywhere, including inside Dynamic Tables.

### STEP 1 & 2 — Create the metric-conversion UDFs

The Pelmorex data is imperial; analysts want metric. Each UDF encapsulates a single unit-conversion formula so you can call it like a built-in function anywhere in SQL.

Send each prompt to CoCo one at a time. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> **STEP 1:** *"Create a SQL UDF named FAHRENHEIT_TO_CELSIUS in TASTY_BYTES.ANALYTICS that accepts a NUMBER(35,4) parameter TEMP_F and returns the Celsius equivalent as NUMBER(35,4)."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Create a SQL UDF" | Asks for a reusable SQL function that returns a value, not a one-off query |
| "named `FAHRENHEIT_TO_CELSIUS` in `TASTY_BYTES.ANALYTICS`" | Gives the exact function name and location that the weather transformation will reference |
| "accepts a `NUMBER(35,4)` parameter `TEMP_F`" | Defines the input name and numeric type, so CoCo doesn't have to choose the function signature |
| "returns the Celsius equivalent as `NUMBER(35,4)`" | States the conversion's direction and return type. Check the generated expression converts Fahrenheit to Celsius, not the reverse. |

> **STEP 2:** *"Create a SQL UDF named INCH_TO_MILLIMETER in TASTY_BYTES.ANALYTICS that accepts a NUMBER(35,4) parameter INCH and returns the millimeter equivalent as NUMBER(35,4)."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Create a SQL UDF named `INCH_TO_MILLIMETER` in `TASTY_BYTES.ANALYTICS`" | Creates the second reusable conversion function under the name the weather transformation expects |
| "accepts a `NUMBER(35,4)` parameter `INCH`" | Defines the input as a numeric measurement in inches |
| "returns the millimeter equivalent as `NUMBER(35,4)`" | Specifies the output unit and type. Check that the generated function converts inches to millimeters. |

<details>
<summary>Expected output (check against this)</summary>

```sql
-- STEP 1
CREATE OR REPLACE FUNCTION TASTY_BYTES.ANALYTICS.FAHRENHEIT_TO_CELSIUS(TEMP_F NUMBER(35,4))
  RETURNS NUMBER(35,4)
AS $$ (TEMP_F - 32) * 5/9 $$;

-- STEP 2
CREATE OR REPLACE FUNCTION TASTY_BYTES.ANALYTICS.INCH_TO_MILLIMETER(INCH NUMBER(35,4))
  RETURNS NUMBER(35,4)
AS $$ INCH * 25.4 $$;
```
</details>

### About Dynamic Tables

A **Dynamic Table** is a table whose contents are defined by a query that Snowflake keeps up to date automatically. You write the `SELECT` once and declare a target freshness (`TARGET_LAG`); Snowflake schedules the refresh and, where possible, only reprocesses changed rows (incremental refresh). You get a view's readability with a table's query speed — no pipeline code to write or schedule.

**Not just for "non-real-time" analytics.** `TARGET_LAG` can be as low as **1 minute** (or `DOWNSTREAM`), so Dynamic Tables serve **both batch and near-real-time / low-latency analytics** — you trade freshness against cost by tuning the lag. (For the *lowest*-latency, high-concurrency serving — real-time dashboards, data APIs — Snowflake also offers **Interactive Tables**.) Docs: https://docs.snowflake.com/en/user-guide/dynamic-tables/overview

**Interoperability:** a Dynamic Table can also be a **Dynamic Iceberg Table**, storing results in open **Apache Iceberg** format on cloud storage so engines like Spark and Trino can read it directly — same refresh model. Docs: https://docs.snowflake.com/en/user-guide/dynamic-tables/create-iceberg

The four DTs build in layers: `DAILY_WEATHER_DT` → `WINDSPEED_HAMBURG_DT` and `WEATHER_HAMBURG_DT`; `SALES_HAMBURG_DT` is the sales side. `WEATHER_HAMBURG_DT` + `SALES_HAMBURG_DT` power the Semantic View.

### STEP 3 — DAILY_WEATHER_DT (base weather, full-refresh)

Joins the live `FROSTBYTE_WEATHERSOURCE` share with Hamburg postal codes.

> **Why `REFRESH_MODE = FULL`?** The source is a third-party share (a database you don't own). Snowflake can only enable the change tracking that powers *incremental* refresh on objects you own — so shares use FULL refresh (recompute each cycle).

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.DAILY_WEATHER_DT
  TARGET_LAG = '1 day'
  REFRESH_MODE = FULL
  INITIALIZE = ON_CREATE
  WAREHOUSE = COMPUTE_WH
AS
SELECT
    hd.*,
    TO_VARCHAR(hd.date_valid_std, 'YYYY-MM') AS yyyy_mm,
    pc.city_name AS city,
    'Germany' AS country_desc
FROM FROSTBYTE_WEATHERSOURCE.onpoint_id.history_day hd
JOIN FROSTBYTE_WEATHERSOURCE.onpoint_id.postal_codes pc
    ON pc.postal_code = hd.postal_code
    AND pc.country = hd.country
WHERE pc.city_name = 'Hamburg';

### STEP 4 — WINDSPEED_HAMBURG_DT

This Dynamic Table filters `DAILY_WEATHER_DT` to Hamburg specifically, tracking daily maximum wind speed. It's the intermediate table that isolates Hamburg's weather pattern — critical context for understanding why sales dropped on specific days.

> **Expected message — not an error.** Creating this table (and `WEATHER_HAMBURG_DT`) may report: *"…successfully created. FULL refresh mode was selected because: Change tracking is not supported on dynamic tables with 'FULL' REFRESH_MODE unless the Dynamic Table has FROZEN WHERE constraint specified."* It's because this DT reads from `DAILY_WEATHER_DT`, which is FULL-refresh (it reads a share). A DT built on a FULL-refresh source can't refresh incrementally either, so Snowflake selects FULL for it too. The table was created correctly — it just refreshes in full each cycle.


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.WINDSPEED_HAMBURG_DT
  TARGET_LAG = '1 day'
  WAREHOUSE = COMPUTE_WH
AS
SELECT
    country_desc,
    city,
    date_valid_std,
    MAX(max_wind_speed_100m_mph) AS max_wind_speed_100m_mph
FROM TASTY_BYTES.HARMONIZED.DAILY_WEATHER_DT
WHERE city = 'Hamburg'
GROUP BY country_desc, city, date_valid_std
ORDER BY date_valid_std;

### STEP 5 — WEATHER_HAMBURG_DT (metric conversions)

This is the final weather Dynamic Table — one of the two that power the Semantic View. It aggregates the postal-level data from `DAILY_WEATHER_DT` into one row per date and converts temperature and precipitation to metric units using the UDFs you just created.


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG_DT
  TARGET_LAG = '1 day'
  WAREHOUSE = COMPUTE_WH
AS
SELECT
    date_valid_std,
    MAX(TASTY_BYTES.ANALYTICS.FAHRENHEIT_TO_CELSIUS(avg_temperature_air_2m_f)) AS avg_temperature_celsius,
    MAX(TASTY_BYTES.ANALYTICS.INCH_TO_MILLIMETER(tot_precipitation_in)) AS avg_precipitation_mm,
    MAX(max_wind_speed_100m_mph) AS max_wind_speed_mph
FROM TASTY_BYTES.HARMONIZED.DAILY_WEATHER_DT
WHERE city = 'Hamburg'
GROUP BY date_valid_std;

### STEP 6 — SALES_HAMBURG_DT (Hamburg sales with a date spine)

The sales-side counterpart to `WEATHER_HAMBURG_DT`. It filters sales to Hamburg and adds a **date spine** — a generated row for every calendar day — so days with **zero orders still appear**. Without the spine, gap days would be missing entirely and the Semantic View couldn't detect the sales drop.

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE TASTY_BYTES.HARMONIZED.SALES_HAMBURG_DT
  TARGET_LAG = '1 day'
  WAREHOUSE = COMPUTE_WH
AS
WITH date_spine AS (
    SELECT DATEADD(DAY, SEQ4(), '2019-01-01') AS order_date
    FROM TABLE(GENERATOR(ROWCOUNT => 3000))
)
SELECT
    ds.order_date,
    ZEROIFNULL(SUM(o.price)) AS daily_sales,
    COUNT(o.order_id) AS num_orders
FROM date_spine ds
LEFT JOIN TASTY_BYTES.ANALYTICS.ORDERS_V o
    ON DATE(o.order_ts) = ds.order_date
    AND o.country = 'Germany'
    AND o.primary_city = 'Hamburg'
GROUP BY ds.order_date
ORDER BY ds.order_date;

## Transformation complete → on to Delivery

You now have four Dynamic Tables that stay fresh automatically:

- `DAILY_WEATHER_DT` — Hamburg postal-code weather, full-refresh
- `WINDSPEED_HAMBURG_DT` — Hamburg wind speed over time
- `WEATHER_HAMBURG_DT` — Hamburg weather in metric units (one row/day)
- `SALES_HAMBURG_DT` — Hamburg sales by day, including zero-sales days

**Delivery** makes this data usable by analysts. In the next cells you will:

1. Build a **Semantic View** that describes the sales and weather tables in business terms.
2. **Query** the Semantic View and chart the February 2022 sales gap.
3. Have **CoCo build a Streamlit dashboard** on top of the Semantic View, then (optionally) extend it with **Cortex Analyst** for natural-language questions.

## Delivery — STEP 1: Create the Semantic View with CoCo

A **Semantic View** describes your tables in **business terms**: the dates to group by, the sales and weather values to analyze, and the relationship that connects them. Cortex Analyst, dashboards, and BI tools can all use it as a single, governed definition of the data.

Send this prompt to CoCo. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it. Optionally, copy the SQL into the notebook cell for future reference.

> *"Create a semantic view named HAMBURG_INSIGHTS_SV in TASTY_BYTES.ANALYTICS using SQL. Source tables: TASTY_BYTES.HARMONIZED.SALES_HAMBURG_DT and TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG_DT. Include daily Hamburg sales, order counts, temperature, precipitation, and wind speed as facts, with metrics total_sales, total_orders, avg_temperature, total_precipitation, and max_wind_speed, and clear comments. Use DATE_VALID_STD as the primary key for WEATHER_HAMBURG_DT and do not set a key on SALES_HAMBURG_DT. Define the relationship from SALES_HAMBURG_DT.ORDER_DATE to WEATHER_HAMBURG_DT.DATE_VALID_STD."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "named `HAMBURG_INSIGHTS_SV` in `TASTY_BYTES.ANALYTICS` using SQL" | Names the object and asks for a `CREATE SEMANTIC VIEW` statement you can run right here |
| "Source tables: … `SALES_HAMBURG_DT` and … `WEATHER_HAMBURG_DT`" | Points CoCo at the final transformed tables, not the raw sales tables |
| "facts … with metrics `total_sales`, `total_orders`, …" | Facts are row-level values; metrics are the aggregations analysts will ask for. Exact metric names let the next cells and the dashboard query them. |
| "Use `DATE_VALID_STD` as the primary key … do not set a key on `SALES_HAMBURG_DT`" | Weather has exactly one row per date, so it's the "one" side of the relationship |
| "relationship from `ORDER_DATE` to `DATE_VALID_STD`" | Connects each sales day to that day's weather |

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE SEMANTIC VIEW TASTY_BYTES.ANALYTICS.HAMBURG_INSIGHTS_SV
  TABLES (
    sales AS TASTY_BYTES.HARMONIZED.SALES_HAMBURG_DT
      COMMENT = 'Daily Tasty Bytes food truck sales in Hamburg, including zero-sales days',
    weather AS TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG_DT
      PRIMARY KEY (DATE_VALID_STD)
      COMMENT = 'Daily Hamburg weather in metric units'
  )
  RELATIONSHIPS (
    sales_to_weather AS sales (ORDER_DATE) REFERENCES weather (DATE_VALID_STD)
  )
  FACTS (
    sales.daily_sales_amount AS DAILY_SALES COMMENT = 'Total sales in USD for the day',
    sales.daily_order_count AS NUM_ORDERS COMMENT = 'Number of orders for the day',
    weather.temperature_celsius AS AVG_TEMPERATURE_CELSIUS COMMENT = 'Average air temperature in Celsius',
    weather.precipitation_mm AS AVG_PRECIPITATION_MM COMMENT = 'Total precipitation in millimeters',
    weather.wind_speed_mph AS MAX_WIND_SPEED_MPH COMMENT = 'Maximum wind speed at 100m in mph'
  )
  DIMENSIONS (
    sales.order_date AS ORDER_DATE COMMENT = 'Calendar date of the sales',
    weather.weather_date AS DATE_VALID_STD COMMENT = 'Calendar date of the weather observation'
  )
  METRICS (
    sales.total_sales AS SUM(sales.daily_sales_amount) COMMENT = 'Total sales in USD',
    sales.total_orders AS SUM(sales.daily_order_count) COMMENT = 'Total number of orders',
    weather.avg_temperature AS AVG(weather.temperature_celsius) COMMENT = 'Average temperature in Celsius',
    weather.total_precipitation AS SUM(weather.precipitation_mm) COMMENT = 'Total precipitation in millimeters',
    weather.max_wind_speed AS MAX(weather.wind_speed_mph) COMMENT = 'Highest wind speed in mph'
  )
  COMMENT = 'Hamburg food truck sales and weather, for investigating sales fluctuations';
```
</details>

> If CoCo's output uses different fact or dimension names, that's fine — but keep the five **metric** names and `order_date` exactly as above, because the dashboard queries them by name.

## Delivery — STEP 2: Query the Semantic View

The `SEMANTIC_VIEW()` table function lets you query by **business concepts** (dimensions and metrics) instead of writing joins. Snowflake uses the relationship you defined to join sales and weather for you.

Run the next cell to pull daily sales, orders, and weather for **February 2022**.

In [ ]:
%%sql -r feb_2022
SELECT *
FROM SEMANTIC_VIEW(
    TASTY_BYTES.ANALYTICS.HAMBURG_INSIGHTS_SV
    DIMENSIONS sales.order_date
    METRICS sales.total_sales, sales.total_orders,
            weather.avg_temperature, weather.total_precipitation, weather.max_wind_speed
)
WHERE order_date BETWEEN '2022-02-01' AND '2022-02-28'
ORDER BY order_date;

### Visualize the sales gap

The next cell charts daily sales against maximum wind speed. Look for days where sales drop to **$0** — what was happening with the wind?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feb = feb_2022.copy() if isinstance(feb_2022, pd.DataFrame) else feb_2022.to_pandas()

fig, ax_sales = plt.subplots(figsize=(12, 5))
ax_sales.bar(feb["ORDER_DATE"], feb["TOTAL_SALES"], color="#29B5E8", label="Daily sales (USD)")
ax_sales.set_ylabel("Daily sales (USD)")

ax_wind = ax_sales.twinx()
ax_wind.plot(feb["ORDER_DATE"], feb["MAX_WIND_SPEED"], color="#FF9F36", marker="o", label="Max wind speed (mph)")
ax_wind.set_ylabel("Max wind speed (mph)")

ax_sales.set_title("Hamburg — daily sales vs. max wind speed, February 2022")
fig.legend(loc="upper right")
fig.autofmt_xdate()
plt.show()

print("Zero-order days:")
print(feb.loc[feb["TOTAL_ORDERS"] == 0, ["ORDER_DATE", "MAX_WIND_SPEED", "AVG_TEMPERATURE"]].to_string(index=False))

## Delivery — STEP 3: Build a Streamlit dashboard with CoCo

A chart in a notebook is great for you; analysts need something they can open themselves. **Streamlit in Snowflake** runs Python data apps directly in your account — no servers, and data never leaves Snowflake.

Instead of writing the app yourself, you'll have **CoCo build and deploy it from a single prompt**. The more precise the prompt, the more predictable the result — notice how it pins down the app name, data source, metric names, and layout.

Send this prompt to CoCo. Review the app code CoCo writes, click **Allow** to let it deploy, then open the app (see **Open the dashboard** below).

> *"Build and deploy a simple Streamlit in Snowflake app named HAMBURG_DASHBOARD in TASTY_BYTES.ANALYTICS with query warehouse COMPUTE_WH and title 'Hamburg Sales & Weather'. Deploy it on the warehouse runtime using only pre-installed packages, with no pyproject.toml, requirements file, or external access integration. The app should query the semantic view TASTY_BYTES.ANALYTICS.HAMBURG_INSIGHTS_SV using the SEMANTIC_VIEW() table function with dimension sales.order_date and metrics sales.total_sales, sales.total_orders, and weather.max_wind_speed. Include only: a start and end date picker defaulting to 2022-02-01 and 2022-02-28, one KPI showing the number of zero-order days, a daily sales bar chart, a max wind speed line chart, and a table listing the zero-order days. Use `st.dataframe()` without the `hide_index` argument for compatibility with the Snowflake Streamlit runtime. Use get_active_session() and pass the dates as bind parameters."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Build and deploy … named `HAMBURG_DASHBOARD` in `TASTY_BYTES.ANALYTICS`" | Asks CoCo to both write the code **and** create the Streamlit object. The exact name is what the autograder checks. |
| "query warehouse `COMPUTE_WH`" | The warehouse that runs the app's queries — the default trial warehouse |
| "warehouse runtime … no `pyproject.toml` … no external access integration" | Keeps the deploy simple: the app uses only packages Snowflake already provides, so it needs no dependency file or network access setup |
| "query the semantic view … using the `SEMANTIC_VIEW()` table function" | Makes the app read from your governed business model, not the raw tables |
| The dimension and five metric names | Uses the exact names you defined in STEP 1, so CoCo doesn't guess column names |
| The date picker, one KPI, charts, and table | Demonstrates the core dashboard workflow without unnecessary UI or metrics |
| "Use `get_active_session()` and pass the dates as bind parameters" | The standard way to connect inside Snowflake, and safe SQL that never pastes user input into a query |

> **If CoCo's app errors or you're short on time:** skip to **Fallback — deploy the pre-built dashboard** below and run those two cells. You'll get the same dashboard (plus the Cortex Analyst tab) and still pass the autograder.

### Open the dashboard

1. In Snowsight, navigate to **Projects » Streamlit**.
2. Open **Hamburg Sales & Weather** (`HAMBURG_DASHBOARD`).
3. The date range defaults to February 2022. Check the **zero-order days** count and compare the sales and wind charts — what was happening on the days sales dropped to $0?

A sales gap that lines up with high wind is a strong lead, but correlation alone doesn't prove cause. Check the dates and values in the table.

## Delivery — STEP 4 (optional): Add natural-language questions with Cortex Analyst

**Cortex Analyst** is Snowflake's text-to-SQL engine. It reads your **Semantic View** to understand the data, then turns a plain-English question into SQL. Here you'll ask CoCo to **modify the app it just built** — another common way to work with CoCo.

Send this prompt to CoCo, review the changes, and click **Allow** to redeploy:

> *"Update the HAMBURG_DASHBOARD Streamlit app: put the existing content in a 'Dashboard' tab and add an 'Ask Cortex Analyst' tab. In the new tab, let users type a question, send it to the Cortex Analyst REST API (/api/v2/cortex/analyst/message) with semantic_view set to TASTY_BYTES.ANALYTICS.HAMBURG_INSIGHTS_SV using _snowflake.send_snow_api_request, then display the text answer, show the generated SQL in an expander, and run that SQL to display the results as a table. Redeploy the app when done."*

**What this prompt asks for**

| Part of the prompt | Why it's there |
|---|---|
| "Update the `HAMBURG_DASHBOARD` Streamlit app" | Edits the app in place rather than creating a second one |
| "'Dashboard' tab … 'Ask Cortex Analyst' tab" | Keeps the existing dashboard intact and adds the new feature alongside it |
| "Cortex Analyst REST API … with `semantic_view` set to …" | Points Analyst at the same governed model the dashboard uses |
| "`_snowflake.send_snow_api_request`" | The built-in way for a Streamlit in Snowflake app to call Snowflake APIs securely, with no credentials in code |
| "show the generated SQL … run that SQL" | Makes the answer transparent: you can see exactly what Analyst queried |

Then, on the **Ask Cortex Analyst** tab, try:
- *"What caused the Hamburg sales gap in February 2022?"*
- *"Show daily Hamburg sales and order counts for February 2022. Which dates had zero orders?"*
- *"Were there other high-wind days in February 2022 when Hamburg still recorded sales?"*

This step is optional — the autograder only checks that `HAMBURG_DASHBOARD` exists.

### Fallback — deploy the pre-built dashboard

**Only run these two cells if CoCo's app didn't work or you're short on time.** They deploy a pre-built version of the same dashboard, with the **Dashboard** and **Ask Cortex Analyst** tabs, as `HAMBURG_DASHBOARD` (replacing any app CoCo created).

1. The first cell creates a stage and uploads the app file (`streamlit_app.py`).
2. The second cell creates the Streamlit app from that file.

Then follow **Open the dashboard** above.

In [ ]:
from io import BytesIO
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("CREATE STAGE IF NOT EXISTS TASTY_BYTES.ANALYTICS.STREAMLIT_STAGE").collect()

STREAMLIT_APP = r'''
import datetime
import json

import _snowflake
import streamlit as st
from snowflake.snowpark.context import get_active_session

SEMANTIC_VIEW = "TASTY_BYTES.ANALYTICS.HAMBURG_INSIGHTS_SV"
session = get_active_session()

st.set_page_config(page_title="Hamburg Sales & Weather", layout="wide")
st.title("Hamburg Sales & Weather")
st.caption(f"Powered by the semantic view `{SEMANTIC_VIEW}`")


@st.cache_data(ttl=600)
def load_daily(start, end):
    return session.sql(
        f"""
        SELECT * FROM SEMANTIC_VIEW(
            {SEMANTIC_VIEW}
            DIMENSIONS sales.order_date
            METRICS sales.total_sales, sales.total_orders, weather.max_wind_speed
        )
        WHERE order_date BETWEEN ? AND ?
        ORDER BY order_date
        """,
        params=[str(start), str(end)],
    ).to_pandas()


dashboard_tab, analyst_tab = st.tabs(["Dashboard", "Ask Cortex Analyst"])

with dashboard_tab:
    col_start, col_end = st.columns(2)
    start = col_start.date_input("Start date", value=datetime.date(2022, 2, 1))
    end = col_end.date_input("End date", value=datetime.date(2022, 2, 28))

    df = load_daily(start, end)
    if df.empty:
        st.warning("No data for the selected dates.")
    else:
        zero_days = df[df["TOTAL_ORDERS"] == 0]
        st.metric("Zero-order days", len(zero_days))

        st.subheader("Daily sales")
        st.bar_chart(df, x="ORDER_DATE", y="TOTAL_SALES")

        st.subheader("Max wind speed (mph)")
        st.line_chart(df, x="ORDER_DATE", y="MAX_WIND_SPEED")

        st.subheader("Zero-order days")
        st.dataframe(zero_days, use_container_width=True)

with analyst_tab:
    st.write("Ask a question about Hamburg sales and weather. Cortex Analyst turns it into SQL against the semantic view.")
    question = st.text_input("Question", value="What caused the Hamburg sales gap in February 2022?")
    if st.button("Ask") and question:
        body = {
            "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
            "semantic_view": SEMANTIC_VIEW,
        }
        with st.spinner("Cortex Analyst is thinking..."):
            resp = _snowflake.send_snow_api_request(
                "POST", "/api/v2/cortex/analyst/message", {}, {}, body, None, 50000
            )
        if resp["status"] >= 400:
            st.error(f"Cortex Analyst returned an error ({resp['status']}): {resp['content']}")
        else:
            content = json.loads(resp["content"])["message"]["content"]
            for item in content:
                if item["type"] == "text":
                    st.markdown(item["text"])
                elif item["type"] == "sql":
                    with st.expander("Generated SQL"):
                        st.code(item["statement"], language="sql")
                    st.dataframe(session.sql(item["statement"]).to_pandas(), use_container_width=True)
                elif item["type"] == "suggestions":
                    st.write("Try asking:")
                    for s in item["suggestions"]:
                        st.write(f"- {s}")
'''

session.file.put_stream(
    BytesIO(STREAMLIT_APP.encode("utf-8")),
    "@TASTY_BYTES.ANALYTICS.STREAMLIT_STAGE/streamlit_app.py",
    auto_compress=False,
    overwrite=True,
)
print("streamlit_app.py uploaded to @TASTY_BYTES.ANALYTICS.STREAMLIT_STAGE")

In [ ]:
CREATE OR REPLACE STREAMLIT TASTY_BYTES.ANALYTICS.HAMBURG_DASHBOARD
  FROM '@TASTY_BYTES.ANALYTICS.STREAMLIT_STAGE'
  MAIN_FILE = 'streamlit_app.py'
  QUERY_WAREHOUSE = COMPUTE_WH
  TITLE = 'Hamburg Sales & Weather';

Come back to this notebook when you're done with the dashboard — the autograder is at the bottom.

## Art of the possible — Cortex Agents and Snowflake CoWork

The Semantic View you built is the foundation for more than dashboards. In environments where they're enabled, the same Semantic View can power:

- **Cortex Agents** — an AI orchestration layer that routes questions to tools like Cortex Analyst (structured data) and Cortex Search (documents), and can chain multiple steps.
- **Snowflake CoWork** — a conversational workspace where business users ask questions of agents, explore follow-ups, and get charts without writing SQL.

You don't need either for this lab. To see what that experience looks like, read the **Deliver With Cortex Agent and CoWork** section of the [public guide](https://www.snowflake.com/en/developers/guides/data-ingestion-transformation-delivery-coco/).

Learn more:
- [Semantic views](https://docs.snowflake.com/en/user-guide/views-semantic/overview)
- [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
- [Streamlit in Snowflake](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit)

## Autograder — earn your badge

Run the next three cells **in order** to submit your lab for the Northstar badge.

1. **Enter your details.** In the `grader_details` cell, replace the placeholders with your information. Use the **same email you used to register** for the workshop and this trial account. Leave `MIDDLE_NAME` as `""` if you don't have one.
2. **Set up the autograder.** Run `grader_setup`. You should see `AUTO_GRADER_IS_WORKING` and a greeting with your name.
3. **Run the answer key.** Run `answer_key`. Every row should show **PASS**. If a step fails, the description tells you which cell to re-run.

> Run the autograder **before** teardown — it checks the objects you created in this lab.

In [ ]:
# Replace these with your details. Use the same email you registered with.
EMAIL = "your.email@company.com"
FIRST_NAME = "First"
MIDDLE_NAME = ""  # leave as "" if you don't have one
LAST_NAME = "Last"

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

assert EMAIL != "your.email@company.com", "Update EMAIL in the grader_details cell first."
assert FIRST_NAME != "First" and LAST_NAME != "Last", "Update your name in the grader_details cell first."

for stmt in [
    "USE ROLE ACCOUNTADMIN",
    """CREATE OR REPLACE API INTEGRATION dora_api_integration
         API_PROVIDER = aws_api_gateway
         API_AWS_ROLE_ARN = 'arn:aws:iam::321463406630:role/snowflakeLearnerAssumedRole'
         ENABLED = TRUE
         API_ALLOWED_PREFIXES = ('https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora')""",
    "CREATE DATABASE IF NOT EXISTS util_db",
    """CREATE OR REPLACE EXTERNAL FUNCTION util_db.public.grader(
         step VARCHAR, passed BOOLEAN, actual INTEGER, expected INTEGER, description VARCHAR)
       RETURNS VARIANT
       API_INTEGRATION = dora_api_integration
       CONTEXT_HEADERS = (current_timestamp, current_account, current_statement, current_account_name)
       AS 'https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora/grader'""",
    """CREATE OR REPLACE EXTERNAL FUNCTION util_db.public.greeting(
         email VARCHAR, firstname VARCHAR, middlename VARCHAR, lastname VARCHAR)
       RETURNS VARIANT
       API_INTEGRATION = dora_api_integration
       CONTEXT_HEADERS = (current_timestamp, current_account, current_statement, current_account_name)
       AS 'https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora/greeting'""",
]:
    session.sql(stmt).collect()

print(session.sql("""
    SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
    FROM (SELECT 'AUTO_GRADER_IS_WORKING' AS step, 123 AS actual, 123 AS expected,
                 'The Snowflake auto-grader has been successfully set up in your account!' AS description)
""").collect()[0][0])

print(session.sql(
    "SELECT util_db.public.greeting(?, ?, ?, ?)",
    params=[EMAIL.strip(), FIRST_NAME.strip(), MIDDLE_NAME.strip(), LAST_NAME.strip()],
).collect()[0][0])

In [ ]:
-- RBC ANSWER KEY — do not modify. Run AFTER the autograder setup cell and BEFORE teardown.
-- Run this script AS-IS after completing all steps in the lab, including the Delivery section.
-- DO NOT modify any queries in this script.

USE ROLE ACCOUNTADMIN;

-- ── PRE-CHECK ── Confirm the autograder is set up ──────────────────────────
EXECUTE IMMEDIATE $$
DECLARE
  autograder_not_set_up EXCEPTION (-20002, 'ACTION REQUIRED: The Snowflake autograder is not set up in this account. Run the autograder setup cell in the notebook first (including the greeting with the same email you used to register), then run this answer key again from top to bottom.');
  grader_count INTEGER DEFAULT 0;
BEGIN
  BEGIN
    SHOW EXTERNAL FUNCTIONS LIKE 'GRADER' IN SCHEMA util_db.public;
    grader_count := (SELECT COUNT(*) FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())));
  EXCEPTION
    WHEN OTHER THEN
      grader_count := 0;
  END;
  IF (grader_count = 0) THEN
    RAISE autograder_not_set_up;
  END IF;
END;
$$;

USE WAREHOUSE COMPUTE_WH;

-- ── PRE-CHECK ── Confirm the lab database exists ───────────────────────────
EXECUTE IMMEDIATE $$
DECLARE
  lab_not_completed EXCEPTION (-20001, 'ACTION REQUIRED: The TASTY_BYTES database was not found in this account. Complete every step of the lab before running this answer key, and make sure you are signed in to the same Snowflake account you used for the lab. Then run this full script again from top to bottom.');
BEGIN
  IF ((SELECT COUNT(*) FROM SNOWFLAKE.INFORMATION_SCHEMA.DATABASES WHERE DATABASE_NAME = 'TASTY_BYTES') = 0) THEN
    RAISE lab_not_completed;
  END IF;
END;
$$;

USE DATABASE TASTY_BYTES;

-- Pre-fetch the Streamlit list once for reuse in step 6 and the summary
SHOW STREAMLITS LIKE 'HAMBURG_DASHBOARD' IN SCHEMA TASTY_BYTES.ANALYTICS;
CREATE OR REPLACE TEMPORARY TABLE _lab_streamlit_check AS
  SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

-- ── STEP 1 ── Database setup ───────────────────────────────────────────────
SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
FROM (
  SELECT
    'BWDC01' AS step,
    (SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.SCHEMATA
     WHERE SCHEMA_NAME IN ('RAW_POS', 'HARMONIZED', 'ANALYTICS')) AS actual,
    3 AS expected,
    'TASTY_BYTES database created with RAW_POS, HARMONIZED, and ANALYTICS schemas — if this fails, re-run the setup cell' AS description
);

-- ── STEP 2 ── Marketplace install ──────────────────────────────────────────
SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
FROM (
  SELECT
    'BWDC02' AS step,
    (SELECT COUNT(*) FROM SNOWFLAKE.INFORMATION_SCHEMA.DATABASES
     WHERE DATABASE_NAME = 'FROSTBYTE_WEATHERSOURCE') AS actual,
    1 AS expected,
    'FROSTBYTE_WEATHERSOURCE Marketplace listing installed — if this fails, re-run the weather_install cell' AS description
);

-- ── STEP 3 ── Data ingestion ───────────────────────────────────────────────
SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
FROM (
  SELECT
    'BWDC03' AS step,
    IFF(
      (SELECT COUNT(*) FROM TASTY_BYTES.RAW_POS.COUNTRY) = 30
      AND (SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLES
           WHERE TABLE_SCHEMA = 'RAW_POS' AND TABLE_TYPE = 'BASE TABLE') = 7,
      1, 0) AS actual,
    1 AS expected,
    'All raw POS tables loaded (COUNTRY: 30 rows, all 7 tables populated) — if this fails, re-run the copy_country and copy_bulk cells' AS description
);

-- ── STEP 4 ── SQL UDFs (existence + correctness check) ────────────────────
SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
FROM (
  SELECT
    'BWDC04' AS step,
    IFF(
      (SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.FUNCTIONS
       WHERE FUNCTION_NAME IN ('FAHRENHEIT_TO_CELSIUS', 'INCH_TO_MILLIMETER')
         AND FUNCTION_SCHEMA = 'ANALYTICS') = 2
      AND TASTY_BYTES.ANALYTICS.FAHRENHEIT_TO_CELSIUS(32) = 0
      AND TASTY_BYTES.ANALYTICS.INCH_TO_MILLIMETER(1) = 25.4,
      1, 0) AS actual,
    1 AS expected,
    'FAHRENHEIT_TO_CELSIUS and INCH_TO_MILLIMETER UDFs created in TASTY_BYTES.ANALYTICS with correct logic — if this fails, re-run the udfs cell' AS description
);

-- ── STEP 5 ── Dynamic Tables (existence + data check) ──────────────────────
SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
FROM (
  SELECT
    'BWDC05' AS step,
    IFF(
      (SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLES
       WHERE TABLE_SCHEMA = 'HARMONIZED'
         AND TABLE_NAME IN ('DAILY_WEATHER_DT', 'WINDSPEED_HAMBURG_DT',
                            'WEATHER_HAMBURG_DT', 'SALES_HAMBURG_DT')) = 4
      AND (SELECT COUNT(*) FROM TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG_DT) > 0
      AND (SELECT COUNT(*) FROM TASTY_BYTES.HARMONIZED.SALES_HAMBURG_DT) > 0,
      1, 0) AS actual,
    1 AS expected,
    'All 4 Dynamic Tables created and populated in TASTY_BYTES.HARMONIZED — if this fails, re-run the DT cells and wait for the initial refresh to complete' AS description
);

-- ── STEP 6 ── Delivery — Streamlit dashboard ───────────────────────────────
SELECT util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
FROM (
  SELECT
    'BWDC06' AS step,
    IFF(
      (SELECT COUNT(*) FROM _lab_streamlit_check
       WHERE "name" = 'HAMBURG_DASHBOARD') = 1,
      1, 0) AS actual,
    1 AS expected,
    'HAMBURG_DASHBOARD Streamlit app created in TASTY_BYTES.ANALYTICS — if this fails, re-run the STEP 3 CoCo prompt or the fallback cells' AS description
);

-- ── FINAL SUMMARY ─────────────────────────────────────────────────────────
-- Self-contained block: rebuilds its own lookup table, submits every step to
-- the grader, AND reports overall status. Running only this block still
-- records all steps, so the "Congratulations" message can never appear
-- without a grader submission.
EXECUTE IMMEDIATE $$
DECLARE
  res RESULTSET;
  msg STRING;
BEGIN
  SHOW STREAMLITS LIKE 'HAMBURG_DASHBOARD' IN SCHEMA TASTY_BYTES.ANALYTICS;
  CREATE OR REPLACE TEMPORARY TABLE _lab_streamlit_check AS
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

  res := (
WITH
check_results AS (
  SELECT 'BWDC01' AS step,
    'TASTY_BYTES database created with RAW_POS, HARMONIZED, and ANALYTICS schemas — if this fails, re-run the setup cell' AS description,
    (SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.SCHEMATA
     WHERE SCHEMA_NAME IN ('RAW_POS', 'HARMONIZED', 'ANALYTICS')) AS actual,
    3 AS expected
  UNION ALL
  SELECT 'BWDC02',
    'FROSTBYTE_WEATHERSOURCE Marketplace listing installed — if this fails, re-run the weather_install cell',
    (SELECT COUNT(*) FROM SNOWFLAKE.INFORMATION_SCHEMA.DATABASES
     WHERE DATABASE_NAME = 'FROSTBYTE_WEATHERSOURCE'),
    1
  UNION ALL
  SELECT 'BWDC03',
    'All raw POS tables loaded (COUNTRY: 30 rows, all 7 tables populated) — if this fails, re-run the copy_country and copy_bulk cells',
    IFF((SELECT COUNT(*) FROM TASTY_BYTES.RAW_POS.COUNTRY) = 30
        AND (SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLES
             WHERE TABLE_SCHEMA = 'RAW_POS' AND TABLE_TYPE = 'BASE TABLE') = 7, 1, 0),
    1
  UNION ALL
  SELECT 'BWDC04',
    'FAHRENHEIT_TO_CELSIUS and INCH_TO_MILLIMETER UDFs created in TASTY_BYTES.ANALYTICS with correct logic — if this fails, re-run the udfs cell',
    IFF((SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.FUNCTIONS
         WHERE FUNCTION_NAME IN ('FAHRENHEIT_TO_CELSIUS', 'INCH_TO_MILLIMETER')
           AND FUNCTION_SCHEMA = 'ANALYTICS') = 2
        AND TASTY_BYTES.ANALYTICS.FAHRENHEIT_TO_CELSIUS(32) = 0
        AND TASTY_BYTES.ANALYTICS.INCH_TO_MILLIMETER(1) = 25.4, 1, 0),
    1
  UNION ALL
  SELECT 'BWDC05',
    'All 4 Dynamic Tables created and populated in TASTY_BYTES.HARMONIZED — if this fails, re-run the DT cells and wait for the initial refresh to complete',
    IFF((SELECT COUNT(*) FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLES
         WHERE TABLE_SCHEMA = 'HARMONIZED'
           AND TABLE_NAME IN ('DAILY_WEATHER_DT', 'WINDSPEED_HAMBURG_DT',
                              'WEATHER_HAMBURG_DT', 'SALES_HAMBURG_DT')) = 4
        AND (SELECT COUNT(*) FROM TASTY_BYTES.HARMONIZED.WEATHER_HAMBURG_DT) > 0
        AND (SELECT COUNT(*) FROM TASTY_BYTES.HARMONIZED.SALES_HAMBURG_DT) > 0, 1, 0),
    1
  UNION ALL
  SELECT 'BWDC06',
    'HAMBURG_DASHBOARD Streamlit app created in TASTY_BYTES.ANALYTICS — if this fails, re-run the STEP 3 CoCo prompt or the fallback cells',
    IFF((SELECT COUNT(*) FROM _lab_streamlit_check
         WHERE "name" = 'HAMBURG_DASHBOARD') = 1, 1, 0),
    1
),
graded AS (
  SELECT step, description, (actual = expected) AS passed,
         util_db.public.grader(step, (actual = expected), actual, expected, description) AS graded_results
  FROM check_results
)
SELECT
  CASE
    WHEN SUM(IFF(passed, 0, 1)) = 0
    THEN 'Congratulations! You have successfully completed the Snowflake Northstar — Data Engineering with CoCo lab!'
    ELSE 'Not all steps passed. Failed: ' ||
         LISTAGG(CASE WHEN NOT passed THEN step END, ', ')
           WITHIN GROUP (ORDER BY step)
  END AS STATUS,
  -- Referencing graded_results keeps the grader call from being optimized away
  COUNT(graded_results) AS STEPS_SUBMITTED
FROM graded
  );
  RETURN TABLE(res);
EXCEPTION
  -- Lab objects are missing (or this block ran before the lab was completed).
  -- Return a readable message instead of a raw SQL compilation error.
  WHEN OTHER THEN
    msg := 'ACTION REQUIRED: Please be sure to run the full script from top to bottom, not just the last line. To do this: (1) click anywhere inside the notebook cell, (2) select all of the script with Cmd+A (Mac) or Ctrl+A (Windows), and (3) click Run. If you still see this message after running the full script, make sure you have completed every step of the lab.';
    -- If the grader itself is missing, point to the setup cell instead.
    BEGIN
      SHOW EXTERNAL FUNCTIONS LIKE 'GRADER' IN SCHEMA util_db.public;
      IF ((SELECT COUNT(*) FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))) = 0) THEN
        msg := 'ACTION REQUIRED: The Snowflake autograder is not set up in this account. Run the autograder setup cell in the notebook first (including the greeting with the same email you used to register), then run this answer key again from top to bottom.';
      END IF;
    EXCEPTION
      WHEN OTHER THEN
        msg := 'ACTION REQUIRED: The Snowflake autograder is not set up in this account. Run the autograder setup cell in the notebook first (including the greeting with the same email you used to register), then run this answer key again from top to bottom.';
    END;
    res := (SELECT :msg AS STATUS, 0 AS STEPS_SUBMITTED);
    RETURN TABLE(res);
END;
$$;

## Teardown

> **Before running teardown:** make sure you've run the **autograder** cells above and every step shows **PASS**. The autograder checks for objects created during this lab — dropping the databases first will cause it to fail.

Once you're done with the lab, run the cell below to remove all objects and stop ongoing credit consumption. The Dynamic Tables in this pipeline refresh automatically — they will continue consuming credits until dropped.

> **Note:** Dropping `TASTY_BYTES` cascades to everything inside it (schemas, tables, dynamic tables, UDFs, semantic view, Streamlit app, and stage). You don't need to drop individual objects.

In [ ]:
-- TEARDOWN — remove all lab objects and stop credit consumption
USE ROLE ACCOUNTADMIN;

-- Drops all schemas, tables, dynamic tables, UDFs, semantic view, Streamlit app, and stage
DROP DATABASE IF EXISTS TASTY_BYTES;

-- Removes the Marketplace weather share
DROP DATABASE IF EXISTS FROSTBYTE_WEATHERSOURCE;

-- Removes the Git API integration created for the workspace
DROP API INTEGRATION IF EXISTS GITHUB_SNOWFLAKE_LABS;